# Polymarket API Exploration

This notebook demonstrates how to interact with the Polymarket API and explore market data.

## Contents
1. Setup and Configuration
2. Fetching Market Data
3. Analyzing Market Characteristics
4. Order Book Analysis
5. Identifying Trading Opportunities
6. Data Persistence Demo

## 1. Setup and Configuration

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from src.api.client import PolymarketClient
from src.data.persistence import MarketDataStore
from src.data.models import Market, OrderBook, Trade, PriceSnapshot
from src.utils.config import load_config

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

print("Imports loaded successfully")

In [ ]:
# Initialize Polymarket client and data store
client = PolymarketClient()
store = MarketDataStore()

# Check API connectivity
if client.health_check():
    print("API connection successful!")
    server_time = client.get_server_time()
    print(f"Server time: {server_time}")
else:
    print("Warning: API health check failed")

## 2. Fetching Market Data

In [ ]:
# Fetch active markets
print("Fetching active markets...")
markets_data = client.get_all_markets(active=True)
print(f"Retrieved {len(markets_data)} active markets")

# Convert to Market objects and save to database
markets = [Market.from_api_response(m) for m in markets_data]
store.save_markets(markets)

# Convert to DataFrame for analysis
df_markets = pd.DataFrame([m.to_dict() for m in markets])
print(f"\nMarket DataFrame shape: {df_markets.shape}")
df_markets.head()

In [ ]:
# Market overview statistics
print("=" * 60)
print("MARKET OVERVIEW")
print("=" * 60)
print(f"Total active markets: {len(df_markets)}")
print(f"Total volume: ${df_markets['volume'].sum():,.2f}")
print(f"Total 24h volume: ${df_markets['volume_24h'].sum():,.2f}")
print(f"Total liquidity: ${df_markets['liquidity'].sum():,.2f}")
print(f"Average volume per market: ${df_markets['volume'].mean():,.2f}")
print(f"Median volume per market: ${df_markets['volume'].median():,.2f}")

## 3. Analyzing Market Characteristics

In [ ]:
# Volume distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale histogram of volumes
ax1 = axes[0]
volumes = df_markets['volume'][df_markets['volume'] > 0]
ax1.hist(np.log10(volumes + 1), bins=50, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Log10(Volume + 1)')
ax1.set_ylabel('Number of Markets')
ax1.set_title('Distribution of Market Volumes (Log Scale)')

# Cumulative volume by top markets
ax2 = axes[1]
sorted_volumes = df_markets.nlargest(50, 'volume')['volume']
cumulative = sorted_volumes.cumsum() / sorted_volumes.sum() * 100
ax2.bar(range(len(cumulative)), cumulative, alpha=0.7)
ax2.axhline(y=80, color='r', linestyle='--', label='80% threshold')
ax2.set_xlabel('Top N Markets')
ax2.set_ylabel('Cumulative Volume %')
ax2.set_title('Cumulative Volume Share by Top Markets')
ax2.legend()

plt.tight_layout()
plt.show()

# Find how many markets make up 80% of volume
for i, pct in enumerate(cumulative):
    if pct >= 80:
        print(f"\nTop {i+1} markets account for 80% of total volume")
        break

In [ ]:
# Top 10 markets by volume
top_markets = df_markets.nlargest(10, 'volume')[[
    'question', 'volume', 'volume_24h', 'liquidity'
]].copy()
top_markets['volume'] = top_markets['volume'].apply(lambda x: f"${x:,.0f}")
top_markets['volume_24h'] = top_markets['volume_24h'].apply(lambda x: f"${x:,.0f}")
top_markets['liquidity'] = top_markets['liquidity'].apply(lambda x: f"${x:,.0f}")

print("\nTop 10 Markets by Total Volume:")
display(top_markets)

In [ ]:
# Liquidity vs Volume analysis
fig, ax = plt.subplots(figsize=(10, 6))

# Filter to markets with both volume and liquidity
df_liq = df_markets[(df_markets['volume'] > 0) & (df_markets['liquidity'] > 0)].copy()

ax.scatter(
    np.log10(df_liq['volume'] + 1),
    np.log10(df_liq['liquidity'] + 1),
    alpha=0.5,
    s=50
)
ax.set_xlabel('Log10(Volume)')
ax.set_ylabel('Log10(Liquidity)')
ax.set_title('Market Volume vs Liquidity')

# Add correlation
corr = df_liq['volume'].corr(df_liq['liquidity'])
ax.text(0.05, 0.95, f'Correlation: {corr:.3f}', transform=ax.transAxes, 
        fontsize=12, verticalalignment='top')

plt.tight_layout()
plt.show()

## 4. Order Book Analysis

In [ ]:
# Select top markets to analyze order books
top_n = 5
top_market_list = df_markets.nlargest(top_n, 'volume')

order_book_data = []

for idx, market in top_market_list.iterrows():
    tokens = market['tokens']
    for token in tokens:
        token_id = token.get('token_id')
        outcome = token.get('outcome', 'Unknown')
        
        if not token_id:
            continue
            
        try:
            # Fetch order book
            book_raw = client.get_order_book(token_id)
            order_book = OrderBook.from_api_response(token_id, book_raw)
            
            # Save to database
            store.save_order_book(order_book)
            
            order_book_data.append({
                'market': market['question'][:40],
                'outcome': outcome,
                'token_id': token_id[:16] + '...',
                'best_bid': order_book.best_bid,
                'best_ask': order_book.best_ask,
                'midpoint': order_book.midpoint,
                'spread': order_book.spread,
                'spread_pct': order_book.spread_pct,
                'bid_depth': order_book.total_bid_size,
                'ask_depth': order_book.total_ask_size,
            })
            
        except Exception as e:
            print(f"Error fetching order book: {e}")

df_books = pd.DataFrame(order_book_data)
print(f"Fetched {len(df_books)} order books")
df_books

In [ ]:
# Spread analysis
if len(df_books) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Spread distribution
    ax1 = axes[0]
    valid_spreads = df_books['spread_pct'].dropna()
    if len(valid_spreads) > 0:
        ax1.hist(valid_spreads, bins=20, edgecolor='black', alpha=0.7)
        ax1.axvline(x=valid_spreads.mean(), color='r', linestyle='--', 
                   label=f'Mean: {valid_spreads.mean():.2f}%')
        ax1.set_xlabel('Spread (%)')
        ax1.set_ylabel('Count')
        ax1.set_title('Bid-Ask Spread Distribution')
        ax1.legend()
    
    # Depth comparison
    ax2 = axes[1]
    x = range(len(df_books))
    width = 0.35
    ax2.bar([i - width/2 for i in x], df_books['bid_depth'], width, label='Bid Depth', alpha=0.7)
    ax2.bar([i + width/2 for i in x], df_books['ask_depth'], width, label='Ask Depth', alpha=0.7)
    ax2.set_xlabel('Token')
    ax2.set_ylabel('Depth (shares)')
    ax2.set_title('Order Book Depth by Token')
    ax2.legend()
    ax2.set_xticks(x)
    ax2.set_xticklabels([f"{row['outcome'][:8]}" for _, row in df_books.iterrows()], rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("No order book data available")

## 5. Identifying Trading Opportunities

In [ ]:
# Find markets with wide spreads (potential market making opportunities)
MIN_SPREAD_PCT = 3.0  # 3% minimum spread
MIN_LIQUIDITY = 1000  # Minimum liquidity in USD

if len(df_books) > 0:
    wide_spread_markets = df_books[
        (df_books['spread_pct'] > MIN_SPREAD_PCT) &
        (df_books['bid_depth'] > MIN_LIQUIDITY) &
        (df_books['ask_depth'] > MIN_LIQUIDITY)
    ].copy()
    
    wide_spread_markets = wide_spread_markets.sort_values('spread_pct', ascending=False)
    
    print(f"Found {len(wide_spread_markets)} tokens with spread > {MIN_SPREAD_PCT}%")
    print("\nPotential Market Making Opportunities:")
    display(wide_spread_markets[['market', 'outcome', 'midpoint', 'spread_pct', 'bid_depth', 'ask_depth']])
else:
    print("No order book data available for analysis")

In [ ]:
# Check for pricing inconsistencies (sum of probabilities should equal 1)
print("Checking for pricing inconsistencies...\n")

for idx, market in top_market_list.head(5).iterrows():
    tokens = market['tokens']
    midpoints = []
    
    for token in tokens:
        token_id = token.get('token_id')
        if token_id:
            try:
                book_raw = client.get_order_book(token_id)
                order_book = OrderBook.from_api_response(token_id, book_raw)
                if order_book.midpoint:
                    midpoints.append({
                        'outcome': token.get('outcome'),
                        'midpoint': order_book.midpoint
                    })
            except:
                pass
    
    if len(midpoints) >= 2:
        total_prob = sum(m['midpoint'] for m in midpoints)
        deviation = abs(1.0 - total_prob)
        
        print(f"Market: {market['question'][:50]}...")
        for m in midpoints:
            print(f"  {m['outcome']}: {m['midpoint']:.4f}")
        print(f"  Sum: {total_prob:.4f} (deviation: {deviation*100:.2f}%)")
        
        if deviation > 0.02:  # More than 2% deviation
            print("  *** POTENTIAL ARBITRAGE OPPORTUNITY ***")
        print()

## 6. Data Persistence Demo

In [ ]:
# Database statistics
stats = store.get_stats()

print("=" * 50)
print("DATABASE STATISTICS")
print("=" * 50)
for key, value in stats.items():
    print(f"{key}: {value}")

In [ ]:
# Export data to CSV
markets_csv = store.export_markets_csv('markets_export.csv')
print(f"Exported markets to: {markets_csv}")

# Read back and display
df_exported = pd.read_csv(markets_csv)
print(f"\nExported {len(df_exported)} markets")
df_exported.head()

## Summary

This notebook demonstrated:
1. Connecting to Polymarket APIs (CLOB and Gamma)
2. Fetching and analyzing market data
3. Order book analysis for trading insights
4. Identifying potential trading opportunities
5. Data persistence for historical analysis

### Key Findings
- Market volume is highly concentrated in a few top markets
- Bid-ask spreads vary significantly across markets
- Some markets show pricing inconsistencies that could present arbitrage opportunities

### Next Steps
- Set up continuous data collection with `scripts/collect_data.py`
- Build historical price charts
- Develop and backtest trading strategies